# 🔷 STEP 4: FEATURE ENGINEERING
Calculate Remaining Useful Life (RUL) and prepare features for modeling

In [0]:
from pyspark.sql.functions import col, max, desc

# Load cleaned data
engine_df = spark.read.table("cmapss_cleaned_data")

print("✅ Data loaded")
print(f"Shape: {engine_df.count()} rows, {len(engine_df.columns)} columns")
display(engine_df.limit(5))

✅ Data loaded
Shape: 160359 rows, 26 columns


engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s17,s18,s19,s20,s21,dataset
87,58,19.9981,0.7004,100.0,491.19,607.01,1484.67,1250.9,9.35,13.65,336.5,2324.02,8725.95,1.08,44.18,316.53,2388.1,8066.86,9.1512,364,2324,100.0,24.67,14.7624,FD004
87,60,10.0011,0.25,100.0,489.05,604.73,1494.44,1295.78,10.52,15.49,396.56,2318.87,8784.51,1.26,45.11,373.31,2388.17,8137.21,8.5954,370,2319,100.0,28.78,17.2833,FD004
87,61,10.0011,0.2512,100.0,489.05,603.8,1497.48,1299.05,10.52,15.48,396.71,2318.91,8779.23,1.26,45.16,373.26,2388.09,8140.73,8.558,368,2319,100.0,28.52,17.1565,FD004
87,67,10.0043,0.25,100.0,489.05,604.85,1497.94,1300.48,10.52,15.49,397.03,2318.91,8782.35,1.26,45.35,373.75,2388.09,8132.87,8.5688,367,2319,100.0,28.78,17.3155,FD004
87,74,20.0045,0.7019,100.0,491.19,607.0,1479.33,1251.85,9.35,13.65,337.22,2324.01,8739.29,1.08,44.32,317.05,2388.21,8071.23,9.1357,364,2324,100.0,24.57,14.7025,FD004


## 1️⃣ Calculate RUL (Remaining Useful Life)

### Understanding RUL
**RUL = Maximum Cycle - Current Cycle**

* RUL tells us how many cycles remain before failure
* This is our **target variable** for prediction
* Example: If engine fails at cycle 200, at cycle 150 the RUL = 50

In [0]:
# Calculate max cycle (failure point) for each engine
max_cycles = engine_df.groupBy("engine_id", "dataset").agg(
    max("cycle").alias("max_cycle")
)

print("📊 Max cycles per engine (sample):")
display(max_cycles.orderBy("engine_id").limit(10))

# Join back to original data
engine_df = engine_df.join(max_cycles, on=["engine_id", "dataset"], how="left")

# Calculate RUL
engine_df = engine_df.withColumn("RUL", col("max_cycle") - col("cycle"))

print("\n✅ RUL calculated successfully")
print(f"Columns now: {engine_df.columns}")

📊 Max cycles per engine (sample):


engine_id,dataset,max_cycle
1,FD003,259
1,FD002,149
1,FD001,192
1,FD004,321
2,FD003,253
2,FD004,299
2,FD001,287
2,FD002,269
3,FD003,222
3,FD004,307



✅ RUL calculated successfully
Columns now: ['engine_id', 'dataset', 'cycle', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's18', 's19', 's20', 's21', 'max_cycle', 'RUL']


## 2️⃣ Validate RUL Calculation

In [0]:
# Verify RUL calculation
print("📊 RUL Statistics:")
engine_df.select("RUL").describe().show()

# Check specific engine
sample = engine_df.filter(col("engine_id") == 1).select("cycle", "max_cycle", "RUL").orderBy("cycle")
print("\n🔍 Sample Engine #1 RUL progression:")
display(sample)

# Verify RUL decreases with cycle
print("\n✅ RUL should decrease from max_cycle to 0")

📊 RUL Statistics:
+-------+------------------+
|summary|               RUL|
+-------+------------------+
|  count|            160359|
|   mean|122.33133781078705|
| stddev| 83.53814553238468|
|    min|                 0|
|    max|               542|
+-------+------------------+


🔍 Sample Engine #1 RUL progression:


cycle,max_cycle,RUL
1,149,148
1,192,191
1,259,258
1,321,320
2,149,147
2,192,190
2,259,257
2,321,319
3,149,146
3,192,189



✅ RUL should decrease from max_cycle to 0


## 3️⃣ Visualize RUL

In [0]:
# Visualize RUL distribution
print("📈 RUL Distribution across all engines:")
display(engine_df.select("RUL"))

# Visualize RUL vs Cycle for sample engine
print("\n📈 RUL degradation for Engine #1:")
display(engine_df.filter(col("engine_id") == 1).select("cycle", "RUL"))

📈 RUL Distribution across all engines:


RUL
201
199
198
192
185
170
148
141
140
83



📈 RUL degradation for Engine #1:


cycle,RUL
22,237
31,228
67,192
69,190
82,177
94,165
95,164
189,70
222,37
231,28


## 4️⃣ Prepare Features for Modeling

In [0]:
# Identify feature columns (sensors and operating conditions)
feature_cols = [c for c in engine_df.columns if c.startswith('s') or c.startswith('op')]

print(f"📊 Feature columns ({len(feature_cols)}): {feature_cols}")

# Select features and target
modeling_df = engine_df.select(
    "engine_id", 
    "dataset", 
    "cycle", 
    "RUL",
    *feature_cols
)

print(f"\n✅ Modeling dataset prepared")
print(f"Shape: {modeling_df.count()} rows, {len(modeling_df.columns)} columns")
display(modeling_df.limit(5))

📊 Feature columns (23): ['op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's18', 's19', 's20', 's21']

✅ Modeling dataset prepared
Shape: 160359 rows, 27 columns


engine_id,dataset,cycle,RUL,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s17,s18,s19,s20,s21
87,FD004,58,201,19.9981,0.7004,100.0,491.19,607.01,1484.67,1250.9,9.35,13.65,336.5,2324.02,8725.95,1.08,44.18,316.53,2388.1,8066.86,9.1512,364,2324,100.0,24.67,14.7624
87,FD004,60,199,10.0011,0.25,100.0,489.05,604.73,1494.44,1295.78,10.52,15.49,396.56,2318.87,8784.51,1.26,45.11,373.31,2388.17,8137.21,8.5954,370,2319,100.0,28.78,17.2833
87,FD004,61,198,10.0011,0.2512,100.0,489.05,603.8,1497.48,1299.05,10.52,15.48,396.71,2318.91,8779.23,1.26,45.16,373.26,2388.09,8140.73,8.558,368,2319,100.0,28.52,17.1565
87,FD004,67,192,10.0043,0.25,100.0,489.05,604.85,1497.94,1300.48,10.52,15.49,397.03,2318.91,8782.35,1.26,45.35,373.75,2388.09,8132.87,8.5688,367,2319,100.0,28.78,17.3155
87,FD004,74,185,20.0045,0.7019,100.0,491.19,607.0,1479.33,1251.85,9.35,13.65,337.22,2324.01,8739.29,1.08,44.32,317.05,2388.21,8071.23,9.1357,364,2324,100.0,24.57,14.7025


## 5️⃣ Feature Normalization (StandardScaler)

### Why Normalize?
* Different sensors have different scales
* Machine learning algorithms perform better with normalized features
* StandardScaler: (x - mean) / std

In [0]:
from pyspark.sql.functions import array

# Assemble features into array (alternative to VectorAssembler for Spark Connect)
df_assembled = modeling_df.withColumn(
    "features_raw",
    array(*[col(c) for c in feature_cols])
)

print("✅ Features assembled into array")
print("New column: features_raw")
display(df_assembled.select("engine_id", "cycle", "RUL", "features_raw").limit(3))

✅ Features assembled into array
New column: features_raw


engine_id,cycle,RUL,features_raw
87,58,201,"List(19.9981, 0.7004, 100.0, 491.19, 607.01, 1484.67, 1250.9, 9.35, 13.65, 336.5, 2324.02, 8725.95, 1.08, 44.18, 316.53, 2388.1, 8066.86, 9.1512, 364.0, 2324.0, 100.0, 24.67, 14.7624)"
87,60,199,"List(10.0011, 0.25, 100.0, 489.05, 604.73, 1494.44, 1295.78, 10.52, 15.49, 396.56, 2318.87, 8784.51, 1.26, 45.11, 373.31, 2388.17, 8137.21, 8.5954, 370.0, 2319.0, 100.0, 28.78, 17.2833)"
87,61,198,"List(10.0011, 0.2512, 100.0, 489.05, 603.8, 1497.48, 1299.05, 10.52, 15.48, 396.71, 2318.91, 8779.23, 1.26, 45.16, 373.26, 2388.09, 8140.73, 8.558, 368.0, 2319.0, 100.0, 28.52, 17.1565)"


In [0]:
from pyspark.sql.functions import mean, stddev_pop, transform

# Manual standardization for Spark Connect compatibility
# Calculate mean and std for each feature position
stats = {}
for i, col_name in enumerate(feature_cols):
    stats_row = df_assembled.select(
        mean(col(col_name)).alias('mean'),
        stddev_pop(col(col_name)).alias('std')
    ).first()
    stats[i] = {'mean': stats_row['mean'], 'std': stats_row['std']}

# Create standardization expression
from pyspark.sql.functions import expr

# Build array of standardized values
standardize_exprs = []
for i, col_name in enumerate(feature_cols):
    mean_val = stats[i]['mean']
    std_val = stats[i]['std'] if stats[i]['std'] > 0 else 1.0
    standardize_exprs.append(f"({col_name} - {mean_val}) / {std_val}")

df_scaled = df_assembled.withColumn(
    "features_scaled",
    array(*[expr(e) for e in standardize_exprs])
)

print("✅ Features normalized using manual standardization")
display(df_scaled.select("engine_id", "cycle", "RUL", "features_raw", "features_scaled").limit(3))

✅ Features normalized using manual standardization


engine_id,cycle,RUL,features_raw,features_scaled
87,58,201,"List(19.9981, 0.7004, 100.0, 491.19, 607.01, 1484.67, 1250.9, 9.35, 13.65, 336.5, 2324.02, 8725.95, 1.08, 44.18, 316.53, 2388.1, 8066.86, 9.1512, 364.0, 2324.0, 100.0, 24.67, 14.7624)","List(0.16857076112426023, 0.7892553994427264, 0.34595473187563586, 0.17584018486377373, 0.2271503130984329, 0.14922245271209197, -0.07378180704332313, -0.12776793949920573, -0.12025869842268859, -0.13340336795997704, 0.3523951624053925, 0.1291752090166979, -0.5186735666824644, -0.009353853140428397, -0.13557109537693335, 0.34591914056328893, -0.2740033352979333, 0.12833406584287246, 0.106417, 0.352572, 0.34595473187576614, -0.10885865959710121, -0.11451108508476648)"
87,60,199,"List(10.0011, 0.25, 100.0, 489.05, 604.73, 1494.44, 1295.78, 10.52, 15.49, 396.56, 2318.87, 8784.51, 1.26, 45.11, 373.31, 2388.17, 8137.21, 8.5954, 370.0, 2319.0, 100.0, 28.78, 17.2833)","List(-0.4362839221641548, -0.4348668899977957, 0.34595473187563586, 0.10549240816400314, 0.17347595489032594, 0.23189652863168012, 0.2554927048798951, 0.1465231645554433, 0.16528256507758143, 0.21150483017127675, 0.3162360771156846, 0.28547847223848427, 0.7480176988503303, 0.2620734737789633, 0.2102415208731155, 0.346548824423486, 0.5985763887441078, -0.6111759977541821, 0.299832, 0.317487, 0.34595473187576614, 0.24268222677000867, 0.24484512278479212)"
87,61,198,"List(10.0011, 0.2512, 100.0, 489.05, 603.8, 1497.48, 1299.05, 10.52, 15.48, 396.71, 2318.91, 8779.23, 1.26, 45.16, 373.26, 2388.09, 8140.73, 8.558, 368.0, 2319.0, 100.0, 28.52, 17.1565)","List(-0.4362839221641548, -0.4316054629388956, 0.34595473187563586, 0.10549240816400314, 0.15158246667385952, 0.2576211131552774, 0.27948396276734877, 0.1465231645554433, 0.1637307103846452, 0.21236623925751852, 0.3165169243800704, 0.27138555506274753, 0.7480176988503303, 0.2766663408176394, 0.2099370011335504, 0.34582918572612026, 0.6422363834651817, -0.660937919989752, 0.235361, 0.317487, 0.34595473187576614, 0.22044363055213784, 0.22676968656841712)"


## 6️⃣ Create Rolling Window Features (Advanced)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

# Create window specification
window_spec = Window.partitionBy("engine_id").orderBy("cycle").rowsBetween(-5, 0)

# Calculate rolling average for key sensors (last 5 cycles)
df_windowed = df_scaled.withColumn("s2_rolling_avg", avg("s2").over(window_spec)) \
                       .withColumn("s3_rolling_avg", avg("s3").over(window_spec)) \
                       .withColumn("s4_rolling_avg", avg("s4").over(window_spec))

print("✅ Rolling window features added")
print("New columns: s2_rolling_avg, s3_rolling_avg, s4_rolling_avg")
display(df_windowed.select("engine_id", "cycle", "s2", "s2_rolling_avg", "RUL").limit(10))

✅ Rolling window features added
New columns: s2_rolling_avg, s3_rolling_avg, s4_rolling_avg


engine_id,cycle,s2,s2_rolling_avg,RUL
1,1,642.36,642.36,258
1,1,555.32,598.84,148
1,1,549.68,582.4533333333334,320
1,1,641.82,597.2950000000001,191
1,2,642.5,606.336,257
1,2,549.9,596.9300000000001,147
1,2,606.07,590.8816666666668,319
1,2,642.15,605.3533333333334,190
1,3,642.18,620.7700000000001,256
1,3,537.31,603.3516666666667,146


## 7️⃣ Save Engineered Features

In [0]:
# Save the feature-engineered dataset
df_windowed.write.format("delta").mode("overwrite").saveAsTable("cmapss_features")

print("✅ Feature-engineered data saved as Delta table: cmapss_features")
print(f"\nFinal dataset shape: {df_windowed.count()} rows, {len(df_windowed.columns)} columns")
print(f"Columns: {df_windowed.columns}")

# Summary statistics
print("\n📊 RUL Summary:")
df_windowed.select("RUL").describe().show()

✅ Feature-engineered data saved as Delta table: cmapss_features

Final dataset shape: 160359 rows, 32 columns
Columns: ['engine_id', 'dataset', 'cycle', 'RUL', 'op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's18', 's19', 's20', 's21', 'features_raw', 'features_scaled', 's2_rolling_avg', 's3_rolling_avg', 's4_rolling_avg']

📊 RUL Summary:
+-------+------------------+
|summary|               RUL|
+-------+------------------+
|  count|            160359|
|   mean|122.33133781078705|
| stddev| 83.53814553238468|
|    min|                 0|
|    max|               542|
+-------+------------------+



## 📊 Feature Engineering Summary

### Accomplishments:
✅ **RUL Calculated**: Target variable for prediction  
✅ **Features Selected**: Sensors (s1-s21) and operating conditions (op1-op3)  
✅ **Normalization**: StandardScaler applied for better model performance  
✅ **Rolling Features**: Window averages to capture temporal patterns  
✅ **Data Saved**: Ready for modeling as `cmapss_features`

### Key Features Created:
* **RUL**: Primary target variable
* **features_scaled**: Normalized sensor readings
* **Rolling averages**: Temporal smoothing for key sensors

**Next Notebook**: `05_Modeling` for PCA, Clustering, and ML